# Long Call / Put Planner

You tell it **which way, how far, how fast, and where you're wrong.** It pulls the live chain and finds the
contracts with the best reward for the risk — then sizes the trade at 1% of your account.

**How to use it:**
1. Fill in Step 2 (every line marked `# <- EDIT`)
2. **Runtime → Run all**
3. Read Steps 3–5

**Remember:** Yahoo data is about 15 minutes delayed. Confirm the contract's mark and IV in thinkorswim before buying.

## Step 1 — Load the tools

In [ ]:
!pip install yfinance -q
import yfinance as yf
import pandas as pd
import numpy as np
from math import log, sqrt, exp, erf, pi
from datetime import date, datetime

def N(x): return 0.5 * (1 + erf(x / sqrt(2)))
def n(x): return exp(-x * x / 2) / sqrt(2 * pi)

def bs(S, K, days, iv, kind, r=0.04):
    """Option value and greeks. At expiration, returns intrinsic value."""
    if days <= 0:
        return {"price": max(S - K, 0) if kind == "call" else max(K - S, 0), "delta": 0.0, "theta": 0.0}
    T = days / 365
    d1 = (log(S / K) + (r + iv**2 / 2) * T) / (iv * sqrt(T))
    d2 = d1 - iv * sqrt(T)
    if kind == "call":
        price = S * N(d1) - K * exp(-r * T) * N(d2)
        delta = N(d1)
        theta = (-S * n(d1) * iv / (2 * sqrt(T)) - r * K * exp(-r * T) * N(d2)) / 365
    else:
        price = K * exp(-r * T) * N(-d2) - S * N(-d1)
        delta = N(d1) - 1
        theta = (-S * n(d1) * iv / (2 * sqrt(T)) + r * K * exp(-r * T) * N(-d2)) / 365
    return {"price": price, "delta": delta, "theta": theta}

print("Tools loaded.")

## Step 2 — Your trade idea

Decide these **before** looking at any option prices. That's the discipline that makes long options work.

In [ ]:
TICKER          = "NVDA"   # <- EDIT
DIRECTION       = "call"   # <- EDIT: "call" if you expect it to rise, "put" if you expect it to fall
TARGET_PRICE    = 230.00   # <- EDIT: where you expect the stock to go
STOP_PRICE      = 205.00   # <- EDIT: the stock price that proves you wrong
DAYS_TO_TARGET  = 15       # <- EDIT: calendar days you expect the move to take
ACCOUNT         = 50_000   # <- EDIT: account size
RISK_PCT        = 0.01     # 1% of the account at risk per trade

MIN_DAYS = max(2 * DAYS_TO_TARGET, 30)   # rule: double the time you think you need, never under 30 days
MAX_DAYS = MIN_DAYS + 45
RISK_DOLLARS = ACCOUNT * RISK_PCT

tk = yf.Ticker(TICKER)
hist = tk.history(period="3mo")
S = float(hist["Close"].iloc[-1])
rv = float(hist["Close"].pct_change().tail(20).std() * np.sqrt(252))

print(f"{TICKER} at ${S:,.2f}  ·  {DIRECTION.upper()}  ·  target ${TARGET_PRICE:,.2f}  ·  stop ${STOP_PRICE:,.2f}  ·  {DAYS_TO_TARGET} days")
move_pct = (TARGET_PRICE / S - 1) * 100
stop_pct = (STOP_PRICE / S - 1) * 100
print(f"You expect a {move_pct:+.1f}% move, and you're wrong at {stop_pct:+.1f}%.")
if (DIRECTION == "call" and not (STOP_PRICE < S < TARGET_PRICE)) or \
   (DIRECTION == "put" and not (TARGET_PRICE < S < STOP_PRICE)):
    print("⚠ Check your inputs: for a call, stop < price < target. For a put, target < price < stop.")
print(f"Looking at expirations {MIN_DAYS}–{MAX_DAYS} days out. Max risk: ${RISK_DOLLARS:,.0f}.")

earnings = None
try:
    cal = tk.calendar
    ed = cal.get("Earnings Date") if isinstance(cal, dict) else None
    earnings = pd.Timestamp(ed[0] if isinstance(ed, (list, tuple)) else ed).date() if ed else None
    if earnings: print(f"Next earnings (per Yahoo): {earnings}")
except Exception:
    pass

## Step 3 — Are options cheap or expensive to buy right now?

Compares what the market is charging (implied vol) to how much the stock is actually moving (realized vol).

In [ ]:
today = date.today()
exps = [e for e in tk.options if MIN_DAYS <= (datetime.strptime(e, "%Y-%m-%d").date() - today).days <= MAX_DAYS]
if not exps:
    print("No expirations in that window — try a different DAYS_TO_TARGET.")
else:
    first = tk.option_chain(exps[0])
    side = first.calls if DIRECTION == "call" else first.puts
    atm = side.iloc[(side["strike"] - S).abs().argsort()[:1]]
    atm_iv = float(atm["impliedVolatility"].iloc[0])
    ratio = atm_iv / rv if rv > 0 else float("nan")
    print(f"At-the-money IV: {atm_iv:.1%}   ·   realized (last 20 days): {rv:.1%}   ·   ratio {ratio:.2f}")
    if ratio > 1.3:
        print("EXPENSIVE to buy — you're paying well above what the stock is actually doing.")
        print("Consider a debit spread instead: the short leg sells some of that rich premium back.")
    elif ratio < 1.0:
        print("CHEAP to buy — options are priced below the stock's actual movement. Good conditions for buyers.")
    else:
        print("Fairly priced — no strong edge either way from volatility.")

## Step 4 — Every contract, scored against your idea

For each contract it estimates:
- **Value at target** — if the stock hits your target in your timeframe
- **Value at stop** — if it hits your stop partway through
- **Value if flat** — if nothing happens and you just lose time

Then it ranks by **reward ÷ risk** and sizes each at 1% of your account.

In [ ]:
rows = []
for e in exps:
    dte = (datetime.strptime(e, "%Y-%m-%d").date() - today).days
    chain = tk.option_chain(e)
    side = chain.calls if DIRECTION == "call" else chain.puts
    for _, o in side.iterrows():
        K, iv = float(o["strike"]), float(o.get("impliedVolatility", 0) or 0)
        bid, ask = float(o.get("bid", 0) or 0), float(o.get("ask", 0) or 0)
        if iv <= 0.01 or ask <= 0:
            continue
        mid = (bid + ask) / 2 if bid > 0 else ask
        now = bs(S, K, dte, iv, DIRECTION)
        if not (0.45 <= abs(now["delta"]) <= 0.75):
            continue
        at_target = bs(TARGET_PRICE, K, dte - DAYS_TO_TARGET, iv, DIRECTION)["price"]
        at_stop   = bs(STOP_PRICE,   K, dte - DAYS_TO_TARGET // 2, iv, DIRECTION)["price"]
        if_flat   = bs(S,            K, dte - DAYS_TO_TARGET, iv, DIRECTION)["price"]
        gain, loss = at_target - mid, mid - at_stop
        if loss <= 0:
            continue
        contracts = int(RISK_DOLLARS // (loss * 100))
        rows.append({
            "Expiration": e, "Days": dte, "Strike": K, "Cost": mid, "Delta": now["delta"], "IV": iv,
            "Value at target": at_target, "Value at stop": at_stop, "Value if flat": if_flat,
            "Reward/Risk": gain / loss, "Contracts": contracts,
            "Cost of trade": contracts * mid * 100, "Max loss at stop": contracts * loss * 100,
            "Gain at target": contracts * gain * 100,
            "Theta/day": now["theta"] * 100,
            "Open interest": int(o.get("openInterest", 0) or 0),
            "Spread %": (ask - bid) / mid if bid > 0 else np.nan,
            "Earnings inside": bool(earnings and today <= earnings <= datetime.strptime(e, "%Y-%m-%d").date()),
        })

scored = pd.DataFrame(rows)
good = pd.DataFrame()
over_limit = False
if scored.empty:
    print("No contracts between 0.45 and 0.75 delta in this window.")
else:
    scored["Loss per contract"] = (scored["Cost"] - scored["Value at stop"]) * 100
    liquid = scored[(scored["Open interest"] >= 100) & ((scored["Spread %"] <= 0.10) | scored["Spread %"].isna())]
    if liquid.empty:
        print("No contracts with enough open interest and a tight enough market. Try another expiration window.")
    else:
        good = liquid[liquid["Contracts"] >= 1].sort_values(["Earnings inside", "Reward/Risk"], ascending=[True, False])
        if good.empty:
            over_limit = True
            cheapest = liquid["Loss per contract"].min()
            print(f"⚠ Even the cheapest contract would lose about ${cheapest:,.0f} if the stock hits your stop —")
            print(f"  more than your ${RISK_DOLLARS:,.0f} risk limit. Your options:")
            print("    • Use a debit spread (buy this strike, sell one near your target) — it costs far less")
            print("    • Move your stop closer — only if a closer level genuinely proves you wrong")
            print("    • Skip the trade")
            print("\nShowing the best contracts at 1 contract each so you can see the numbers:\n")
            good = liquid.sort_values(["Earnings inside", "Reward/Risk"], ascending=[True, False]).copy()
            good["Contracts"] = 1
            good["Cost of trade"] = good["Cost"] * 100
            good["Max loss at stop"] = good["Loss per contract"]
            good["Gain at target"] = (good["Value at target"] - good["Cost"]) * 100
        view = good.head(10)[["Expiration", "Days", "Strike", "Cost", "Delta", "Reward/Risk", "Contracts",
                              "Cost of trade", "Max loss at stop", "Gain at target", "Theta/day", "Earnings inside"]].copy()
        view["Cost"] = view["Cost"].map("${:,.2f}".format)
        for col in ("Cost of trade", "Max loss at stop", "Gain at target", "Theta/day"):
            view[col] = view[col].map("${:,.0f}".format)
        view["Delta"] = view["Delta"].map("{:+.2f}".format)
        view["Reward/Risk"] = view["Reward/Risk"].map("{:.2f}".format)
        print(f"Top contracts for your {DIRECTION} idea:\n")
        print(view.to_string(index=False))
        print("\nRule of thumb: only take it if Reward/Risk is at least 1.5.")
        best_rr = good["Reward/Risk"].max()
        stock_rr = abs(TARGET_PRICE - S) / abs(S - STOP_PRICE)
        if best_rr < 1.5:
            print(f"\n⚠ Nothing reaches 1.5. On the stock itself, your idea makes {abs(TARGET_PRICE/S-1):.1%} "
                  f"and risks {abs(STOP_PRICE/S-1):.1%} — only {stock_rr:.2f} to 1.")
            print("  No option contract can fix a weak target-to-stop ratio. Find a closer stop that genuinely")
            print("  proves you wrong, a bigger realistic target, or skip the trade.")
        if good["Earnings inside"].all():
            print("\n⚠ Every contract here spans an earnings report. Consider waiting until after it.")

## Step 5 — The full plan for the top contract

In [ ]:
if not good.empty:
    r = good.iloc[0]
    k, c, dte = r["Strike"], int(r["Contracts"]), int(r["Days"])
    be = k + r["Cost"] if DIRECTION == "call" else k - r["Cost"]
    if r["Reward/Risk"] < 1.5 or over_limit:
        print("⚠ REFERENCE ONLY — this plan breaks your rules:")
        if r["Reward/Risk"] < 1.5:
            print(f"   • Reward/risk is {r['Reward/Risk']:.2f}, below your 1.5 minimum")
        if over_limit:
            print(f"   • One contract risks ${r['Max loss at stop']:,.0f}, more than your ${RISK_DOLLARS:,.0f} limit")
        print()
    print(f"BUY {c} × {TICKER} {r['Expiration']} {k:g} {DIRECTION.upper()}  at about ${r['Cost']:.2f}  "
          f"(${r['Cost of trade']:,.0f} total)")
    print()
    print(f"  Exit at target  (stock ${TARGET_PRICE:,.2f}):  sell near ${r['Value at target']:.2f}  →  +${r['Gain at target']:,.0f}")
    print(f"  Exit at stop    (stock ${STOP_PRICE:,.2f}):    sell near ${r['Value at stop']:.2f}  →  −${r['Max loss at stop']:,.0f}")
    print(f"  If nothing happens for {DAYS_TO_TARGET} days: worth about ${r['Value if flat']:.2f}  →  "
          f"{'+' if r['Value if flat'] >= r['Cost'] else '−'}${abs(r['Value if flat'] - r['Cost']) * c * 100:,.0f} from time decay")
    print(f"  Time decay right now:  ${abs(r['Theta/day']) * c:,.0f} per day for the position")
    print(f"  Breakeven at expiration: ${be:,.2f}")
    print()
    print("  Value on your target date at different stock prices:")
    prices = np.linspace(min(STOP_PRICE, TARGET_PRICE) * 0.97, max(STOP_PRICE, TARGET_PRICE) * 1.03, 7)
    for p in prices:
        v = bs(p, k, dte - DAYS_TO_TARGET, r["IV"], DIRECTION)["price"]
        pl = (v - r["Cost"]) * c * 100
        print(f"     {TICKER} at ${p:>9,.2f}  →  option ${v:>6.2f}   P&L {'+' if pl >= 0 else '−'}${abs(pl):,.0f}")
    if r["Earnings inside"]:
        print("\n  ⚠ Earnings fall before this expiration — expect IV to drop after the report, which hurts long options.")
    print()
    print("  Time stop: if the stock hasn't moved your way by day", DAYS_TO_TARGET, "— sell and move on.")

## Your rules, every time

1. **Direction from the daily ADX** — calls only with green on top, puts only with red on top.
2. **Delta 0.50–0.70.** The planner only shows 0.45–0.75.
3. **Expiration at least double your expected time,** never under 30 days.
4. **Target, stop and time limit decided first.**
5. **Size so the loss at your stop is 1% of the account.**
6. **Exit at target, stop, or time — never average down.**
7. **If Step 3 says EXPENSIVE,** use a debit spread instead of a naked call or put.